#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possiblily not fit into RAM. It also contains supporting code for evaluation and parameter tuning

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

In [3]:
loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=10)
docs = text_splitter.split_documents(documents)

Created a chunk of size 329, which is longer than the specified 200
Created a chunk of size 314, which is longer than the specified 200
Created a chunk of size 291, which is longer than the specified 200
Created a chunk of size 302, which is longer than the specified 200
Created a chunk of size 334, which is longer than the specified 200
Created a chunk of size 279, which is longer than the specified 200
Created a chunk of size 351, which is longer than the specified 200
Created a chunk of size 293, which is longer than the specified 200
Created a chunk of size 272, which is longer than the specified 200
Created a chunk of size 287, which is longer than the specified 200
Created a chunk of size 310, which is longer than the specified 200
Created a chunk of size 285, which is longer than the specified 200
Created a chunk of size 240, which is longer than the specified 200
Created a chunk of size 225, which is longer than the specified 200
Created a chunk of size 216, which is longer tha

In [5]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
db = FAISS.from_documents(docs, embeddings)
db

In [6]:
### querying
query = "What is essential for conversational AI."
docs = db.similarity_search(query)
docs[0].page_content

'Fourth, Memory is essential for conversational AI. Langchain provides several memory types to maintain conversation history. You can keep track of previous interactions, which is vital for creating chatbots that understand context and can have meaningful, coherent conversations.'

#### As a Retriever
We can also convert the vectorstore into retriever class. This allows us to easily use it in other Langchain methods, which largely work with retrievers.

In [8]:
retriever = db.as_retriever()
docs = retriever.invoke(query)

In [10]:
docs[0].page_content

'Fourth, Memory is essential for conversational AI. Langchain provides several memory types to maintain conversation history. You can keep track of previous interactions, which is vital for creating chatbots that understand context and can have meaningful, coherent conversations.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [11]:
docs_and_score = db.similarity_search_with_score(query)

In [12]:
docs_and_score

[(Document(id='9cc2ada8-b6aa-4622-94e9-d8e0ad661896', metadata={'source': 'speech.txt'}, page_content='Fourth, Memory is essential for conversational AI. Langchain provides several memory types to maintain conversation history. You can keep track of previous interactions, which is vital for creating chatbots that understand context and can have meaningful, coherent conversations.'),
  np.float32(0.44801748)),
 (Document(id='16a08338-33c3-4244-9a83-23811915b548', metadata={'source': 'speech.txt'}, page_content='Chatbots: Build conversational agents that understand context, remember previous interactions, and can perform actions by using tools and APIs.'),
  np.float32(0.5070794)),
 (Document(id='55ce60da-4770-4015-9b6d-5b42afab252f', metadata={'source': 'speech.txt'}, page_content='Begin with simple applications - a basic chatbot, a Q&A system over your documents. Understand the core concepts: models, prompts, chains, and memory. Then explore more advanced features like agents and integ

In [13]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[0.05267120152711868,
 0.03224026411771774,
 -0.05343537777662277,
 -0.007949634455144405,
 0.020616482943296432,
 -0.042261067777872086,
 -0.021835077553987503,
 -0.007627848535776138,
 0.003589264117181301,
 -0.0030228146351873875,
 0.004644943401217461,
 0.024022120982408524,
 -0.031873881816864014,
 -0.03852035477757454,
 -0.019429517909884453,
 0.012883317656815052,
 -0.04746745154261589,
 0.004298272076994181,
 -0.015336385928094387,
 -0.03277166560292244,
 0.02612994983792305,
 0.007695969194173813,
 -0.05825388804078102,
 0.010560962371528149,
 -0.06592805683612823,
 0.02196182869374752,
 -0.020159399136900902,
 0.03442055732011795,
 0.06863366067409515,
 0.04296847805380821,
 -0.010641279630362988,
 0.007812082767486572,
 -0.01436745934188366,
 -0.04666300117969513,
 -0.0024457203689962626,
 -0.018606409430503845,
 0.015815038233995438,
 -0.021036043763160706,
 -0.047813739627599716,
 -0.015219461172819138,
 0.002417182782664895,
 -0.016400370746850967,
 0.03130945935845375,
 

In [17]:
docs_score = db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='9cc2ada8-b6aa-4622-94e9-d8e0ad661896', metadata={'source': 'speech.txt'}, page_content='Fourth, Memory is essential for conversational AI. Langchain provides several memory types to maintain conversation history. You can keep track of previous interactions, which is vital for creating chatbots that understand context and can have meaningful, coherent conversations.'),
 Document(id='16a08338-33c3-4244-9a83-23811915b548', metadata={'source': 'speech.txt'}, page_content='Chatbots: Build conversational agents that understand context, remember previous interactions, and can perform actions by using tools and APIs.'),
 Document(id='55ce60da-4770-4015-9b6d-5b42afab252f', metadata={'source': 'speech.txt'}, page_content='Begin with simple applications - a basic chatbot, a Q&A system over your documents. Understand the core concepts: models, prompts, chains, and memory. Then explore more advanced features like agents and integrations.'),
 Document(id='8e2954af-af92-4916-9c96-77e5a8

In [18]:
### Saving and loading the vector store
db.save_local("faiss_index")

In [19]:
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

In [20]:
docs = new_db.similarity_search(query)

In [21]:
docs

[Document(id='9cc2ada8-b6aa-4622-94e9-d8e0ad661896', metadata={'source': 'speech.txt'}, page_content='Fourth, Memory is essential for conversational AI. Langchain provides several memory types to maintain conversation history. You can keep track of previous interactions, which is vital for creating chatbots that understand context and can have meaningful, coherent conversations.'),
 Document(id='16a08338-33c3-4244-9a83-23811915b548', metadata={'source': 'speech.txt'}, page_content='Chatbots: Build conversational agents that understand context, remember previous interactions, and can perform actions by using tools and APIs.'),
 Document(id='55ce60da-4770-4015-9b6d-5b42afab252f', metadata={'source': 'speech.txt'}, page_content='Begin with simple applications - a basic chatbot, a Q&A system over your documents. Understand the core concepts: models, prompts, chains, and memory. Then explore more advanced features like agents and integrations.'),
 Document(id='8e2954af-af92-4916-9c96-77e5a8